# Czyszczenie danych 2

W tym notebooku sprawdzam dane i robię drugą wersję czyszczenia. Chodzi głównie o to, żeby późniejsza analiza w `analiza_danych.ipynb` była prostsza i nie musiała za każdym razem poprawiać tych samych problemów.

Po uruchomieniu notebook zapisuje trzy pliki:
- `data/dane_przetworzone_2.csv` - dane po czyszczeniu, jeden wiersz to jeden wypadek,
- `data/dane_pojazdy_przyczyny_2.csv` - pomocniczy plik, gdzie pojazdy i przyczyny są w jednej kolumnie,
- `data/raport_czyszczenia_2.csv` - krótka informacja, ile wierszy zostało usuniętych.


In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)

DATA_DIR = Path("data")
RAW_PATH = DATA_DIR / "dane_projektowe_ny_collisions.csv"
CLEAN_PATH = DATA_DIR / "dane_przetworzone_2.csv"
LONG_PATH = DATA_DIR / "dane_pojazdy_przyczyny_2.csv"
QUALITY_REPORT_PATH = DATA_DIR / "raport_czyszczenia_2.csv"


## Przygotowanie funkcji

Na początku zapisuję listy kolumn, które powtarzają się w danych. W pliku są osobne kolumny dla pojazdu 1, 2, 3, 4 i 5, więc bez tego kod byłby mocno powtarzalny.


In [2]:
FACTOR_COLS = [f"CONTRIBUTING FACTOR VEHICLE {i}" for i in range(1, 6)]
VEHICLE_COLS = [f"VEHICLE TYPE CODE {i}" for i in range(1, 6)]

INJURED_DETAIL_COLS = [
    "NUMBER OF PEDESTRIANS INJURED",
    "NUMBER OF CYCLIST INJURED",
    "NUMBER OF MOTORIST INJURED",
]
KILLED_DETAIL_COLS = [
    "NUMBER OF PEDESTRIANS KILLED",
    "NUMBER OF CYCLIST KILLED",
    "NUMBER OF MOTORIST KILLED",
]
TOTAL_CASUALTY_COLS = [
    "NUMBER OF PERSONS INJURED",
    "NUMBER OF PERSONS KILLED",
    *INJURED_DETAIL_COLS,
    *KILLED_DETAIL_COLS,
]

BOROUGHS = ["BRONX", "BROOKLYN", "MANHATTAN", "QUEENS", "STATEN ISLAND"]
NYC_LAT_RANGE = (40.45, 40.95)
NYC_LON_RANGE = (-74.30, -73.65)

quality_log = []


def log_step(step, before, after, note=""):
    quality_log.append(
        {
            "krok": step,
            "liczba_wierszy_przed": int(before),
            "liczba_wierszy_po": int(after),
            "usuniete_wiersze": int(before - after),
            "opis": note,
        }
    )


def clean_string_series(series):
    return (
        series.astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .replace({"": pd.NA, "NAN": pd.NA, "NONE": pd.NA})
    )


FACTOR_REPLACEMENTS = {
    "1": "UNSPECIFIED",
    "80": "UNSPECIFIED",
    "ILLNES": "ILLNESS",
    "REACTION TO OTHER UNINVOLVED VEHICLE": "REACTION TO UNINVOLVED VEHICLE",
}


def normalize_factor_value(value):
    if pd.isna(value):
        return "EMPTY"
    value = str(value).strip().upper()
    if value in {"", "NAN", "NONE"}:
        return "EMPTY"
    return FACTOR_REPLACEMENTS.get(value, value)


VEHICLE_REPLACEMENTS = {
    "SPORT UTILITY / STATION WAGON": ["STATION WAGON/SPORT UTILITY VEHICLE", "SPORT UTILITY/STATION WAGON"],
    "PASSENGER VEHICLE": ["PASSA", "PASSE", "PASS"],
    "SEDAN": ["2 DR SEDAN", "4 DR SEDAN", "SE", "SEA", "4D", "4DR", "4DS", "4DSD"],
    "PICK-UP TRUCK": ["PICK", "PICK-", "PICKU", "PK"],
    "BOX TRUCK": ["BOX", "BOX T", "BOXTR"],
    "BICYCLE": ["BIKE", "BICYC", "BK"],
    "BUS": ["BU"],
    "VAN": ["VN", "VAB", "VAN T", "VAN/T", "VAN CAMPER", "VANETTE", "VAN`"],
    "TAXI": ["TAXI CAB"],
    "AMBULANCE": ["AM", "AMABU", "AMB", "AMBU", "AMBUL"],
    "FIRE TRUCK": ["FD NY", "FDNY", "FIRE", "FIRET"],
    "MOPED": ["MOPD", "MO-PED", "MOPEN", "MOPET"],
    "SCOOTER": ["SCOO", "SCOOT"],
    "CONVERTIBLE": ["CONV", "CONVE"],
    "DELIVERY": ["DEL", "DELIV", "DELV", "DELVI"],
    "DUMP": ["DP", "DUMPS", "DUMPT"],
    "FLAT BED": ["FLATB"],
    "GARBAGE OR REFUSE": ["GARBA"],
    "TOW TRUCK / WRECKER": ["TOW", "TOW T", "TOW TRUCK", "TOW-T", "TOWTR"],
    "LARGE COM VEH(6 OR MORE TIRES)": ["LADDE"],
    "TRUCK": ["TK"],
}
VEHICLE_LOOKUP = {
    old_value: clean_value
    for clean_value, old_values in VEHICLE_REPLACEMENTS.items()
    for old_value in old_values
}

NOISY_VEHICLE_VALUES = {
    "00", "1", "994", "72000", "BLUE", "BROWN", "RED,", "N/A", "NA", "UNK,", "UNNKO",
    "UNKNOWN", "OTHER", "NYC", "GOV'T", "GOV V", "NEW Y", "U HAU", "U-TRU", "UBER",
    "TOYOT", "CHEVR", "GMC V", "RAM", "CAT", "CAT 3", "CAT 4", "JCB40", "JOHND",
    "BED", "BLOCK", "BUMP", "DIRT-", "DIRTB", "TRIAL", "HOUSE", "PISH", "POST",
}


def normalize_vehicle_value(value):
    if pd.isna(value):
        return pd.NA
    value = str(value).strip().upper()
    if value in VEHICLE_LOOKUP:
        return VEHICLE_LOOKUP[value]
    if value in NOISY_VEHICLE_VALUES:
        return "UNKNOWN/OTHER"
    if len(value) <= 2:
        return "UNKNOWN/OTHER"
    if any(char.isdigit() for char in value) and len(value) <= 5:
        return "UNKNOWN/OTHER"
    return value


def unique_value_lookup(frame, keys, value):
    valid = frame.dropna(subset=keys + [value])
    unique_counts = valid.groupby(keys)[value].nunique()
    unique_index = unique_counts[unique_counts.eq(1)].index
    first_values = valid.groupby(keys)[value].first()
    return first_values.loc[unique_index].to_dict()


## Wczytanie danych

Wczytuję oryginalny plik z danymi. Na tym etapie niczego jeszcze nie usuwam.


In [3]:
raw = pd.read_csv(RAW_PATH, low_memory=False)
log_step("Dane wejściowe", len(raw), len(raw), "Oryginalny plik CSV przed czyszczeniem")

print(f"Liczba wierszy: {len(raw):,}".replace(",", " "))
print(f"Liczba kolumn: {raw.shape[1]}")
display(raw.head())


Liczba wierszy: 1 612 178
Liczba kolumn: 29


,ACCIDENT DATE,ACCIDENT TIME,BOROUGH,ZIP CODE,LATITUDE,LONGITUDE,LOCATION,ON STREET NAME,CROSS STREET NAME,OFF STREET NAME,NUMBER OF PERSONS INJURED,NUMBER OF PERSONS KILLED,NUMBER OF PEDESTRIANS INJURED,NUMBER OF PEDESTRIANS KILLED,NUMBER OF CYCLIST INJURED,NUMBER OF CYCLIST KILLED,NUMBER OF MOTORIST INJURED,NUMBER OF MOTORIST KILLED,CONTRIBUTING FACTOR VEHICLE 1,CONTRIBUTING FACTOR VEHICLE 2,CONTRIBUTING FACTOR VEHICLE 3,CONTRIBUTING FACTOR VEHICLE 4,CONTRIBUTING FACTOR VEHICLE 5,COLLISION_ID,VEHICLE TYPE CODE 1,VEHICLE TYPE CODE 2,VEHICLE TYPE CODE 3,VEHICLE TYPE CODE 4,VEHICLE TYPE CODE 5
0,2019-08-05T00:00:00.000,16:30,QUEENS,11434,40.676052,-73.790184,"{'type': 'Point', 'coordinates': [-73.790184, ...",NaN,NaN,150-08 123 AVENUE,0.0,0.0,0,0,0,0,0,0,Passing Too Closely,Unspecified,NaN,NaN,NaN,4184637,Sedan,Pick-up Truck,NaN,NaN,NaN
1,2019-08-27T00:00:00.000,16:02,BROOKLYN,11225,40.657780,-73.951096,"{'type': 'Point', 'coordinates': [-73.951096, ...",NaN,NaN,288 HAWTHORNE STREET,0.0,0.0,0,0,0,0,0,0,Passing Too Closely,Unspecified,NaN,NaN,NaN,4195773,Station Wagon/Sport Utility Vehicle,Station Wagon/Sport Utility Vehicle,NaN,NaN,NaN
2,2019-08-15T00:00:00.000,17:57,MANHATTAN,10002,40.718143,-73.993835,"{'type': 'Point', 'coordinates': [-73.993835, ...",CHRYSTIE STREET,GRAND STREET,NaN,1.0,0.0,0,0,0,0,1,0,Driver Inattention/Distraction,NaN,NaN,NaN,NaN,4202457,Sedan,NaN,NaN,NaN,NaN
3,2019-08-30T00:00:00.000,21:53,BRONX,10460,40.840534,-73.866610,"{'type': 'Point', 'coordinates': [-73.86661, 4...",NaN,NaN,1837 EAST TREMONT AVENUE,0.0,0.0,0,0,0,0,0,0,Unspecified,Unspecified,NaN,NaN,NaN,4198749,Taxi,Station Wagon/Sport Utility Vehicle,NaN,NaN,NaN
4,2019-08-06T00:00:00.000,9:45,MANHATTAN,10016,40.745440,-73.975400,"{'type': 'Point', 'coordinates': [-73.9754, 40...",EAST 35 STREET,2 AVENUE,NaN,1.0,0.0,0,0,1,0,0,0,Driver Inattention/Distraction,Driver Inattention/Distraction,NaN,NaN,NaN,4183798,Station Wagon/Sport Utility Vehicle,Bike,NaN,NaN,NaN


## Poprawa typów danych i dat

Zmieniam kolumny liczbowe na liczby, a datę i godzinę łączę w jedną kolumnę. Dodaję też rok, miesiąc, godzinę i dzień tygodnia, bo mogą się przydać w analizie.


## Sprawdzenie danych przed czyszczeniem

Zanim czyszczę dane, sprawdzam podstawowe problemy: duplikaty, braki, błędne współrzędne, niespójne sumy rannych i zabitych oraz najczęstsze wartości tekstowe.


In [4]:
raw_profile = pd.DataFrame(
    {
        "metryka": [
            "liczba wierszy",
            "liczba kolumn",
            "unikalne COLLISION_ID",
            "dokładne duplikaty wierszy",
            "rekordy bez dzielnicy",
            "rekordy bez kodu pocztowego",
            "rekordy bez współrzędnych lub z zerami",
        ],
        "wartość": [
            len(raw),
            raw.shape[1],
            raw["COLLISION_ID"].nunique(),
            int(raw.duplicated().sum()),
            int(raw["BOROUGH"].isna().sum()),
            int(raw["ZIP CODE"].isna().sum()),
            int(
                raw["LATITUDE"].isna().sum()
                + raw["LONGITUDE"].isna().sum()
                + ((raw["LATITUDE"] == 0) | (raw["LONGITUDE"] == 0)).sum()
            ),
        ],
    }
)
display(raw_profile)

missing_top = raw.isna().mean().mul(100).sort_values(ascending=False).head(12).rename("% braków")
display(missing_top.to_frame())

raw_injured_check = raw[INJURED_DETAIL_COLS].sum(axis=1)
raw_killed_check = raw[KILLED_DETAIL_COLS].sum(axis=1)
raw_casualty_mismatch = raw["NUMBER OF PERSONS INJURED"].ne(raw_injured_check) | raw["NUMBER OF PERSONS KILLED"].ne(raw_killed_check)
print("Niespójne rekordy obrażeń/zgonów przed czyszczeniem:", int(raw_casualty_mismatch.sum()))

raw_factors = pd.concat([clean_string_series(raw[col]).str.upper() for col in FACTOR_COLS], ignore_index=True).dropna()
raw_vehicles = pd.concat([clean_string_series(raw[col]).str.upper() for col in VEHICLE_COLS], ignore_index=True).dropna()

print("Najczęstsze surowe czynniki:")
display(raw_factors.value_counts().head(10).to_frame("liczba_wystąpień"))

print("Najczęstsze surowe typy pojazdów:")
display(raw_vehicles.value_counts().head(10).to_frame("liczba_wystąpień"))


,metryka,wartość
0,liczba wierszy,1612178
1,liczba kolumn,29
2,unikalne COLLISION_ID,1217957
3,dokładne duplikaty wierszy,394221
4,rekordy bez dzielnicy,484625
5,rekordy bez kodu pocztowego,484802
6,rekordy bez współrzędnych lub z zerami,393796


,% braków
CONTRIBUTING FACTOR VEHICLE 5,99.651279
VEHICLE TYPE CODE 5,99.370417
CONTRIBUTING FACTOR VEHICLE 4,98.648474
VEHICLE TYPE CODE 4,97.146159
CONTRIBUTING FACTOR VEHICLE 3,93.533344
VEHICLE TYPE CODE 3,92.014405
OFF STREET NAME,86.370488
CROSS STREET NAME,33.059935
ZIP CODE,30.071245
BOROUGH,30.060266


Niespójne rekordy obrażeń/zgonów przed czyszczeniem: 403
Najczęstsze surowe czynniki:


,liczba_wystąpień
UNSPECIFIED,1892099
DRIVER INATTENTION/DISTRACTION,374570
FAILURE TO YIELD RIGHT-OF-WAY,105739
FOLLOWING TOO CLOSELY,93248
OTHER VEHICULAR,80424
BACKING UNSAFELY,68500
FATIGUED/DROWSY,60918
TURNING IMPROPERLY,49403
PASSING OR LANE USAGE IMPROPER,46915
PASSING TOO CLOSELY,40725


Najczęstsze surowe typy pojazdów:


,liczba_wystąpień
PASSENGER VEHICLE,1338214
SPORT UTILITY / STATION WAGON,611527
SEDAN,283179
STATION WAGON/SPORT UTILITY VEHICLE,233586
TAXI,129488
UNKNOWN,107270
PICK-UP TRUCK,70070
VAN,59063
OTHER,51274
BUS,38706


### Co poprawiam dalej

Po szybkim sprawdzeniu danych widać kilka rzeczy do poprawy:

- usuwam dokładne duplikaty wierszy,
- usuwam rekordy, gdzie liczba rannych lub zabitych nie zgadza się z sumą z kolumn szczegółowych,
- uzupełniam dzielnicę i kod pocztowy tylko wtedy, gdy da się to zrobić na podstawie innych rekordów,
- oznaczam błędne lub puste współrzędne,
- porządkuję nazwy czynników wypadków i typów pojazdów,
- tworzę osobny plik pomocniczy dla pojazdów i przyczyn.


In [5]:
data = raw.copy()

for col in TOTAL_CASUALTY_COLS:
    data[col] = pd.to_numeric(data[col], errors="coerce").fillna(0).astype("Int64")

data["LATITUDE"] = pd.to_numeric(data["LATITUDE"], errors="coerce")
data["LONGITUDE"] = pd.to_numeric(data["LONGITUDE"], errors="coerce")
data["COLLISION_ID"] = pd.to_numeric(data["COLLISION_ID"], errors="coerce").astype("Int64")

data["ZIP CODE"] = clean_string_series(data["ZIP CODE"]).str.extract(r"(\d{5})", expand=False)
data["BOROUGH"] = clean_string_series(data["BOROUGH"]).str.upper()
data.loc[~data["BOROUGH"].isin(BOROUGHS), "BOROUGH"] = pd.NA

date_part = clean_string_series(data["ACCIDENT DATE"]).str.slice(0, 10)
time_part = clean_string_series(data["ACCIDENT TIME"]).fillna("00:00")
data["ACCIDENT_DATE_CLEAN"] = pd.to_datetime(date_part, errors="coerce")
data["ACCIDENT_DATETIME"] = pd.to_datetime(date_part + " " + time_part, errors="coerce")
data["YEAR"] = data["ACCIDENT_DATE_CLEAN"].dt.year.astype("Int64")
data["MONTH"] = data["ACCIDENT_DATE_CLEAN"].dt.month.astype("Int64")
data["HOUR"] = data["ACCIDENT_DATETIME"].dt.hour.astype("Int64")
data["DAY_OF_WEEK"] = data["ACCIDENT_DATE_CLEAN"].dt.day_name()

print("Zakres dat:", data["ACCIDENT_DATE_CLEAN"].min(), "-", data["ACCIDENT_DATE_CLEAN"].max())


Zakres dat: 2012-07-01 00:00:00 - 2019-11-26 00:00:00


## Usunięcie duplikatów

`COLLISION_ID` powinien oznaczać pojedynczy wypadek. W danych było dużo takich samych wierszy, więc usuwam dokładne powtórzenia.


In [6]:
before = len(data)
exact_duplicates = int(data.duplicated().sum())
data = data.drop_duplicates().copy()
log_step(
    "Usunięcie dokładnych duplikatów",
    before,
    len(data),
    f"Usunięto {exact_duplicates:,} identycznych wierszy".replace(",", " "),
)

remaining_collision_duplicates = int(data["COLLISION_ID"].duplicated().sum())
print("Pozostałe duplikaty COLLISION_ID:", remaining_collision_duplicates)
if remaining_collision_duplicates:
    before = len(data)
    data = data.sort_values("ACCIDENT_DATETIME").drop_duplicates("COLLISION_ID", keep="first")
    log_step("Usunięcie pozostałych duplikatów COLLISION_ID", before, len(data), "Zachowano pierwszy rekord po czasie wypadku")


Pozostałe duplikaty COLLISION_ID: 0


## Sprawdzenie rannych i zabitych

Sprawdzam, czy suma rannych i zabitych w kategoriach pieszy, rowerzysta i kierowca zgadza się z kolumną ogólną. Jeśli nie, taki rekord usuwam.


In [7]:
data["_injured_check"] = data[INJURED_DETAIL_COLS].sum(axis=1)
data["_killed_check"] = data[KILLED_DETAIL_COLS].sum(axis=1)
casualty_mismatch = (
    data["NUMBER OF PERSONS INJURED"].ne(data["_injured_check"])
    | data["NUMBER OF PERSONS KILLED"].ne(data["_killed_check"])
)

print("Niespójne rekordy obrażeń/zgonów:", int(casualty_mismatch.sum()))
display(
    data.loc[
        casualty_mismatch,
        [
            "COLLISION_ID",
            "NUMBER OF PERSONS INJURED",
            "_injured_check",
            "NUMBER OF PERSONS KILLED",
            "_killed_check",
        ],
    ].head()
)

before = len(data)
data = data.loc[~casualty_mismatch].drop(columns=["_injured_check", "_killed_check"]).copy()
log_step("Usunięcie niespójnych rekordów obrażeń/zgonów", before, len(data), "Kontrola sum cząstkowych i łącznych")


Niespójne rekordy obrażeń/zgonów: 323


,COLLISION_ID,NUMBER OF PERSONS INJURED,_injured_check,NUMBER OF PERSONS KILLED,_killed_check
21759,4025300,0,3,0,0
22398,4025229,0,2,0,0
23269,4025759,1,0,0,0
26832,4026042,0,1,0,0
31457,4025398,0,1,0,0


## Dzielnice, kody pocztowe i współrzędne

Braki dzielnic i kodów pocztowych uzupełniam tylko wtedy, gdy w samych danych jest jednoznaczna informacja. Nie używam tutaj zewnętrznych plików geograficznych.


In [8]:
invalid_coords = (
    data["LATITUDE"].isna()
    | data["LONGITUDE"].isna()
    | data["LATITUDE"].eq(0)
    | data["LONGITUDE"].eq(0)
    | ~data["LATITUDE"].between(*NYC_LAT_RANGE)
    | ~data["LONGITUDE"].between(*NYC_LON_RANGE)
)

data["brak_wspolrzednych"] = invalid_coords
data.loc[invalid_coords, ["LATITUDE", "LONGITUDE"]] = pd.NA

data["brak_dzielnicy_przed_uzupelnieniem"] = data["BOROUGH"].isna()
data["brak_kodu_przed_uzupelnieniem"] = data["ZIP CODE"].isna()

zip_to_borough = unique_value_lookup(data, ["ZIP CODE"], "BOROUGH")
coord_to_borough = unique_value_lookup(data, ["LATITUDE", "LONGITUDE"], "BOROUGH")
coord_to_zip = unique_value_lookup(data, ["LATITUDE", "LONGITUDE"], "ZIP CODE")

missing_borough_with_zip = data["BOROUGH"].isna() & data["ZIP CODE"].notna()
data.loc[missing_borough_with_zip, "BOROUGH"] = data.loc[missing_borough_with_zip, "ZIP CODE"].map(zip_to_borough)

missing_borough_with_coords = data["BOROUGH"].isna() & data["LATITUDE"].notna() & data["LONGITUDE"].notna()
coord_keys = list(zip(data.loc[missing_borough_with_coords, "LATITUDE"], data.loc[missing_borough_with_coords, "LONGITUDE"]))
data.loc[missing_borough_with_coords, "BOROUGH"] = [coord_to_borough.get(key, pd.NA) for key in coord_keys]

missing_zip_with_coords = data["ZIP CODE"].isna() & data["LATITUDE"].notna() & data["LONGITUDE"].notna()
coord_keys = list(zip(data.loc[missing_zip_with_coords, "LATITUDE"], data.loc[missing_zip_with_coords, "LONGITUDE"]))
data.loc[missing_zip_with_coords, "ZIP CODE"] = [coord_to_zip.get(key, pd.NA) for key in coord_keys]

data["uzupelniona_dzielnica"] = data["brak_dzielnicy_przed_uzupelnieniem"] & data["BOROUGH"].notna()
data["uzupelniony_kod_pocztowy"] = data["brak_kodu_przed_uzupelnieniem"] & data["ZIP CODE"].notna()
data["brak_dzielnicy"] = data["BOROUGH"].isna()
data["brak_kodu"] = data["ZIP CODE"].isna()

summary_geo = pd.DataFrame(
    {
        "metryka": [
            "brak współrzędnych po walidacji",
            "brak dzielnicy przed uzupełnieniem",
            "uzupełniona dzielnica",
            "brak dzielnicy po uzupełnieniu",
            "brak kodu przed uzupełnieniem",
            "uzupełniony kod pocztowy",
            "brak kodu po uzupełnieniu",
        ],
        "liczba_wierszy": [
            int(data["brak_wspolrzednych"].sum()),
            int(data["brak_dzielnicy_przed_uzupelnieniem"].sum()),
            int(data["uzupelniona_dzielnica"].sum()),
            int(data["brak_dzielnicy"].sum()),
            int(data["brak_kodu_przed_uzupelnieniem"].sum()),
            int(data["uzupelniony_kod_pocztowy"].sum()),
            int(data["brak_kodu"].sum()),
        ],
    }
)
display(summary_geo)


,metryka,liczba_wierszy
0,brak współrzędnych po walidacji,147480
1,brak dzielnicy przed uzupełnieniem,370427
2,uzupełniona dzielnica,93267
3,brak dzielnicy po uzupełnieniu,277160
4,brak kodu przed uzupełnieniem,370591
5,uzupełniony kod pocztowy,91048
6,brak kodu po uzupełnieniu,279543


## Poprawa tekstów i kategorii

Porządkuję nazwy ulic, czynników wypadków i typów pojazdów. Usuwam nadmiarowe spacje, zmieniam tekst na wielkie litery i poprawiam najczęstsze skróty.


In [9]:
for col in ["ON STREET NAME", "CROSS STREET NAME", "OFF STREET NAME"]:
    data[col] = clean_string_series(data[col]).str.upper()

for col in FACTOR_COLS:
    data[col] = clean_string_series(data[col]).str.upper().map(normalize_factor_value).astype("string")

for col in VEHICLE_COLS:
    data[col] = clean_string_series(data[col]).str.upper().map(normalize_vehicle_value).astype("string")

street_pair = data["ON STREET NAME"].notna() & data["CROSS STREET NAME"].notna()
data["miejsce_opisowe"] = data["ON STREET NAME"]
data.loc[street_pair, "miejsce_opisowe"] = data.loc[street_pair, "ON STREET NAME"] + " / " + data.loc[street_pair, "CROSS STREET NAME"]
data["miejsce_opisowe"] = data["miejsce_opisowe"].fillna(data["OFF STREET NAME"]).fillna("BRAK OPISU")

print("Najczęstsze czynniki po normalizacji:")
display(pd.concat([data[col] for col in FACTOR_COLS]).value_counts().head(15).to_frame("liczba_wystąpień"))

print("Najczęstsze typy pojazdów po normalizacji:")
display(pd.concat([data[col] for col in VEHICLE_COLS]).dropna().value_counts().head(15).to_frame("liczba_wystąpień"))


Najczęstsze czynniki po normalizacji:


,liczba_wystąpień
EMPTY,3721063
UNSPECIFIED,1418202
DRIVER INATTENTION/DISTRACTION,287366
FAILURE TO YIELD RIGHT-OF-WAY,80644
FOLLOWING TOO CLOSELY,73531
OTHER VEHICULAR,60793
BACKING UNSAFELY,52219
FATIGUED/DROWSY,44001
TURNING IMPROPERLY,37244
PASSING OR LANE USAGE IMPROPER,36956


Najczęstsze typy pojazdów po normalizacji:


,liczba_wystąpień
PASSENGER VEHICLE,985765
SPORT UTILITY / STATION WAGON,646961
SEDAN,233277
UNKNOWN/OTHER,118951
TAXI,98130
PICK-UP TRUCK,55105
VAN,44135
BICYCLE,33962
BUS,32616
SMALL COM VEH(4 TIRES),22343


## Liczba wypadków w tym samym miejscu

Dla każdej pary współrzędnych liczę, ile razy wystąpił tam wypadek. Ta informacja jest później używana przy analizie miejsc.


In [10]:
location_counts = (
    data.loc[~data["brak_wspolrzednych"]]
    .groupby(["LATITUDE", "LONGITUDE"])
    .size()
    .rename("ile_wypadkow_w_danym_miejscu")
    .reset_index()
)

data = data.merge(location_counts, on=["LATITUDE", "LONGITUDE"], how="left")
data["ile_wypadkow_w_danym_miejscu"] = data["ile_wypadkow_w_danym_miejscu"].fillna(0).astype("Int64")

display(data[["COLLISION_ID", "BOROUGH", "LATITUDE", "LONGITUDE", "miejsce_opisowe", "ile_wypadkow_w_danym_miejscu"]].head())


,COLLISION_ID,BOROUGH,LATITUDE,LONGITUDE,miejsce_opisowe,ile_wypadkow_w_danym_miejscu
0,4184637,QUEENS,40.676052,-73.790184,150-08 123 AVENUE,1
1,4195773,BROOKLYN,40.657780,-73.951096,288 HAWTHORNE STREET,2
2,4202457,MANHATTAN,40.718143,-73.993835,CHRYSTIE STREET / GRAND STREET,73
3,4198749,BRONX,40.840534,-73.866610,1837 EAST TREMONT AVENUE,2
4,4183798,MANHATTAN,40.745440,-73.975400,EAST 35 STREET / 2 AVENUE,83


## Plik pomocniczy dla pojazdów i przyczyn

W oryginalnych danych pojazdy i czynniki są rozbite na pięć kolumn. Tutaj robię z tego prostszy układ: jeden wiersz oznacza jeden pojazd lub czynnik z danego wypadku.


In [11]:
base_cols = [
    "COLLISION_ID",
    "ACCIDENT_DATE_CLEAN",
    "ACCIDENT_DATETIME",
    "YEAR",
    "MONTH",
    "HOUR",
    "DAY_OF_WEEK",
    "BOROUGH",
    "ZIP CODE",
    "LATITUDE",
    "LONGITUDE",
    "miejsce_opisowe",
    "NUMBER OF PERSONS INJURED",
    "NUMBER OF PERSONS KILLED",
]

long_parts = []
for i in range(1, 6):
    factor_col = f"CONTRIBUTING FACTOR VEHICLE {i}"
    vehicle_col = f"VEHICLE TYPE CODE {i}"
    part = data[base_cols + [factor_col, vehicle_col]].copy()
    part = part.rename(columns={factor_col: "CONTRIBUTING FACTOR", vehicle_col: "VEHICLE TYPE"})
    part["NR_POJAZDU"] = i
    part = part[
        part["VEHICLE TYPE"].notna()
        | part["CONTRIBUTING FACTOR"].ne("EMPTY")
    ].copy()
    long_parts.append(part)

vehicle_factor_long = pd.concat(long_parts, ignore_index=True)
vehicle_factor_long["CZYNNIK_OKRESLONY"] = vehicle_factor_long["CONTRIBUTING FACTOR"].ne("EMPTY")
vehicle_factor_long["CZYNNIK_MERYTORYCZNY"] = ~vehicle_factor_long["CONTRIBUTING FACTOR"].isin(["EMPTY", "UNSPECIFIED"])

print("Liczba rekordów w tabeli długiej:", f"{len(vehicle_factor_long):,}".replace(",", " "))
display(vehicle_factor_long.head())


Liczba rekordów w tabeli długiej: 2 435 927


,COLLISION_ID,ACCIDENT_DATE_CLEAN,ACCIDENT_DATETIME,YEAR,MONTH,HOUR,DAY_OF_WEEK,BOROUGH,ZIP CODE,LATITUDE,LONGITUDE,miejsce_opisowe,NUMBER OF PERSONS INJURED,NUMBER OF PERSONS KILLED,CONTRIBUTING FACTOR,VEHICLE TYPE,NR_POJAZDU,CZYNNIK_OKRESLONY,CZYNNIK_MERYTORYCZNY
0,4184637,2019-08-05,2019-08-05 16:30:00,2019,8,16,Monday,QUEENS,11434,40.676052,-73.790184,150-08 123 AVENUE,0,0,PASSING TOO CLOSELY,SEDAN,1,True,True
1,4195773,2019-08-27,2019-08-27 16:02:00,2019,8,16,Tuesday,BROOKLYN,11225,40.657780,-73.951096,288 HAWTHORNE STREET,0,0,PASSING TOO CLOSELY,SPORT UTILITY / STATION WAGON,1,True,True
2,4202457,2019-08-15,2019-08-15 17:57:00,2019,8,17,Thursday,MANHATTAN,10002,40.718143,-73.993835,CHRYSTIE STREET / GRAND STREET,1,0,DRIVER INATTENTION/DISTRACTION,SEDAN,1,True,True
3,4198749,2019-08-30,2019-08-30 21:53:00,2019,8,21,Friday,BRONX,10460,40.840534,-73.866610,1837 EAST TREMONT AVENUE,0,0,UNSPECIFIED,TAXI,1,True,False
4,4183798,2019-08-06,2019-08-06 09:45:00,2019,8,9,Tuesday,MANHATTAN,10016,40.745440,-73.975400,EAST 35 STREET / 2 AVENUE,1,0,DRIVER INATTENTION/DISTRACTION,SPORT UTILITY / STATION WAGON,1,True,True


## Zapis plików

Na końcu zapisuję dane po czyszczeniu oraz plik pomocniczy do dalszej analizy.


In [12]:
quality_report = pd.DataFrame(quality_log)
quality_report.to_csv(QUALITY_REPORT_PATH, index=False, encoding="utf-8")
data.to_csv(CLEAN_PATH, index=False, encoding="utf-8")
vehicle_factor_long.to_csv(LONG_PATH, index=False, encoding="utf-8")

print("Zapisano:")
print("-", CLEAN_PATH)
print("-", LONG_PATH)
print("-", QUALITY_REPORT_PATH)

display(quality_report)


Zapisano:
- data\dane_przetworzone_2.csv
- data\dane_pojazdy_przyczyny_2.csv
- data\raport_czyszczenia_2.csv


,krok,liczba_wierszy_przed,liczba_wierszy_po,usuniete_wiersze,opis
0,Dane wejściowe,1612178,1612178,0,Oryginalny plik CSV przed czyszczeniem
1,Usunięcie dokładnych duplikatów,1612178,1217957,394221,Usunięto 394 221 identycznych wierszy
2,Usunięcie niespójnych rekordów obrażeń/zgonów,1217957,1217634,323,Kontrola sum cząstkowych i łącznych
